# Football Database Setup and Data Loading

Welcome to the Football Database Jupyter Notebook!  
This notebook guides you through the process of setting up a PostgreSQL database for football analytics, including:

- **Defining and creating core tables** (`games`, `players`, `plays`, `week_data`)
- **Populating tables** with data from CSV files
- **Ensuring data integrity** and proper schema design for efficient analysis

> **Prerequisites:**  
> - PostgreSQL server access  
> - Environment variables configured for database connection  
> - Data files available in the `../data/` directory

Follow the steps in each section to prepare your database for advanced football data analysis and visualization.

In [ ]:
# Import required libraries and get the database connection URI for the football database
import psycopg2
from helpers.db_utils import get_football_connection_uri

# Retrieve the connection URI for the football database
# This will be used to connect to the PostgreSQL instance
# Make sure your environment variables are set up as described in the docs/env_file_configuration.md

# Get the connection URI
db_football_uri = get_football_connection_uri()

# Tables Creation

In this section, we define and create the main tables required for the football database: `games`, `players`, `plays`, and `week_data`. Each table is structured to efficiently store relevant information for subsequent analysis. The creation process ensures that all necessary tables exist in the database before loading any data.

In [ ]:
# Define SQL schemas for the main tables: games, players, plays and week data
# These schemas will be used to create the tables if they do not already exist in the database
TABLE_SCHEMAS = {
    "games": """
        CREATE TABLE IF NOT EXISTS games (
            gameId VARCHAR PRIMARY KEY,
            gameDate DATE,
            gameTimeEastern TIME,
            homeTeamAbbr VARCHAR,
            visitorTeamAbbr VARCHAR,
            week INTEGER
        );
    """,
    "players": """
        CREATE TABLE IF NOT EXISTS players (
            nflId VARCHAR PRIMARY KEY,
            height VARCHAR,
            weight INTEGER,
            birthDate DATE,
            collegeName VARCHAR,
            position VARCHAR,
            displayName VARCHAR
        );
    """,
    "plays": """
        CREATE TABLE IF NOT EXISTS plays (
            gameId VARCHAR,
            playId BIGINT,
            playDescription TEXT,
            quarter INTEGER,
            down INTEGER,
            yardsToGo INTEGER,
            possessionTeam VARCHAR,
            playType VARCHAR,
            yardlineSide VARCHAR,
            yardlineNumber BIGINT,
            offenseFormation VARCHAR,
            personnelO VARCHAR,
            defendersInTheBox BIGINT,
            numberOfPassRushers BIGINT,
            personnelD VARCHAR,
            typeDropback VARCHAR,
            preSnapVisitorScore BIGINT,
            preSnapHomeScore BIGINT,
            gameClock TIME,
            absoluteYardlineNumber BIGINT,
            penaltyCodes VARCHAR,
            penaltyJerseyNumbers VARCHAR,
            passResult VARCHAR,
            offensePlayResult BIGINT,
            playResult BIGINT,
            epa FLOAT,
            isDefensivePI BOOLEAN
        );
    """,
    "week_data": """
        CREATE TABLE IF NOT EXISTS week_data (
            time TIMESTAMP,
            x FLOAT,
            y FLOAT,
            s FLOAT,
            a FLOAT,
            dis FLOAT,
            o FLOAT,
            dir FLOAT,
            event VARCHAR,
            nflId VARCHAR,
            displayName VARCHAR,
            jerseyNumber INTEGER,
            position VARCHAR,
            frameId INTEGER,
            team VARCHAR,
            gameId VARCHAR,
            playId INTEGER,
            playDirection VARCHAR,
            route VARCHAR
        );
    """
}

In [ ]:
# Create the Games Table using the schema defined in TABLE_SCHEMAS

# Get the SQL query for creating the games table
games_table_query = TABLE_SCHEMAS["games"]

# Connect to the football database
football_conn = psycopg2.connect(db_football_uri)
cur = football_conn.cursor()

# Execute the table creation query
cur.execute(games_table_query)
football_conn.commit()

# Close the cursor and connection
cur.close()
football_conn.close()

print("Games table created successfully.")

In [ ]:
# Create the Players Table using the schema defined in TABLE_SCHEMAS

# Get the SQL query for creating the players table
players_table_query = TABLE_SCHEMAS["players"]

# Connect to the football database
football_conn = psycopg2.connect(db_football_uri)
cur = football_conn.cursor()

# Execute the table creation query
cur.execute(players_table_query)
football_conn.commit()

# Close the cursor and connection
cur.close()
football_conn.close()

print("Players table created successfully.")

In [ ]:
# Create the Plays Table using the schema defined in TABLE_SCHEMAS

# Get the SQL query for creating the plays table
plays_table_query = TABLE_SCHEMAS["plays"]

# Connect to the football database
football_conn = psycopg2.connect(db_football_uri)
cur = football_conn.cursor()

# Execute the table creation query
cur.execute(plays_table_query)
football_conn.commit()

# Close the cursor and connection
cur.close()
football_conn.close()

print("Plays table created successfully.")

In [ ]:
# Create the Week Data Table using the schema defined in TABLE_SCHEMAS

# Get the SQL query for creating the week_data table
week_data_table_query = TABLE_SCHEMAS["week_data"]

# Connect to the football database
football_conn = psycopg2.connect(db_football_uri)
cur = football_conn.cursor()

# Execute the table creation query
cur.execute(week_data_table_query)
football_conn.commit()

# Close the cursor and connection
cur.close()
football_conn.close()

print("Week data table created successfully.")

# Populating the Tables

In this section, we load data from CSV files and populate the database tables (`games`, `players`, `plays`, and `week_data`). The process ensures that each table receives the appropriate data, handling file extraction and data type conversions as needed. This step is essential for preparing the database for analysis and querying.

In [ ]:
# File paths for each table's data
# These files will be used to populate the corresponding tables in the database
FILES = {
    "games": "../data/games.csv",         # CSV file containing games data
    "players": "../data/players.csv",     # CSV file containing players data
    "plays": "../data/plays.csv",         # CSV file containing plays data
    "week_data": "../data/week_data.csv"  # CSV file containing week data (may also be inside a zip)
}

In [ ]:
import os
import pandas as pd
import zipfile
import re

# Function to read week_data from CSV or ZIP file
def read_week_data():
    # If week_data.csv exists, read it directly
    if os.path.exists("../data/week_data.csv"):
        return pd.read_csv("../data/week_data.csv")
    # Otherwise, look for week_data.zip and extract/read the CSV
    elif os.path.exists("../data/week_data.zip"):
        with zipfile.ZipFile("../data/week_data.zip", "r") as z:
            with z.open("datasets/week_data.csv") as f:
                return pd.read_csv(f)
    else:
        raise FileNotFoundError("Neither week_data.csv nor week_data.zip found.")

# Populate Tables
try:
    # Connect to the football database
    conn = psycopg2.connect(db_football_uri)
    cur = conn.cursor()

    # Load and insert data for each table
    for table, file_path in FILES.items():
        # Special handling for week_data (may be zipped)
        if table == "week_data":
            df = read_week_data()
        else:
            df = pd.read_csv(file_path)
        # Clean column names by stripping whitespace
        df.columns = [col.strip() for col in df.columns]

        # Ensure BIGINT columns are within range and correct dtype
        BIGINT_RANGE = (-9223372036854775808, 9223372036854775807)
        # Find BIGINT columns for this table using regex on schema
        schema = TABLE_SCHEMAS[table]
        bigint_cols = [re.findall(r'(\w+)\s+BIGINT', line)[0]
                    for line in schema.splitlines() if 'BIGINT' in line]
        for col in bigint_cols:
            if col in df.columns:
                # Convert to numeric, coerce errors, clip to BIGINT range, and convert to pandas nullable Int64
                df[col] = pd.to_numeric(df[col], errors='coerce')
                df[col] = df[col].clip(lower=BIGINT_RANGE[0], upper=BIGINT_RANGE[1])
                df[col] = df[col].astype('Int64')

        # Replace pd.NA and np.nan with None for all columns (for SQL compatibility)
        df = df.astype(object).where(pd.notnull(df), None)

        # Prepare SQL for bulk insert
        placeholders = ', '.join(['%s'] * len(df.columns))
        columns = ', '.join(df.columns)
        sql = f"INSERT INTO {table} ({columns}) VALUES ({placeholders}) ON CONFLICT DO NOTHING;"
        # Bulk insert data into the table
        cur.executemany(sql, df.values.tolist())

        print(f"Inserted data into: {table}")
        conn.commit()

    # Close cursor and connection
    cur.close()
    conn.close()
    print("✅ All data inserted successfully.")

except Exception as e:
    # Print any errors encountered during the process
    print("❌ Error:", e)
